# RAG 메트릭 데모 — SDK & API

검색 기반 생성(RAG) 에이전트용 메트릭을 SDK와 API 두 방식으로 계산한다.

대상 메트릭: `recall_at_k`, `precision_at_k`, `ndcg_at_k`, `faithfulness`, `consistency`

- 검색 품질(recall/precision/ndcg): `metadata`의 검색 id와 정답 id를 비교한다 — LLM이 필요 없는 순수 계산이다.
- 근거성(faithfulness/consistency): 응답(`output`)을 검색된 근거(`retrieved_context`)에 대해 **실제 LLM judge**가 채점한다.

## 사전 준비

```bash
uv sync --extra server --extra t2s
```

> 참고: judge는 temperature 0으로 호출하지만 실행마다 점수가 조금 달라질 수 있다.
> 이 노트북 전체 실행 시 judge 호출은 12회다(항목 3 × 메트릭 2 × SDK/API 두 파트).

## 0. 데이터셋

각 항목은 질문, 응답, 검색된 근거 텍스트/id, 정답(relevant) id, 그리고 NDCG용 등급 관련도(`relevance`)를 갖는다.

In [ ]:
DATASET = [
    {
        "input": "회사의 환불 정책은 어떻게 되나요?",
        "output": "구매 후 30일 이내에는 전액 환불이 가능합니다.",
        "expected": "30일 이내 전액 환불",
        "retrieved_context": ["환불 정책: 제품 구매 후 30일 이내에 요청하면 전액 환불됩니다.", "배송은 보통 3~5일 걸립니다."],
        "retrieved_ids": ["doc1", "doc7", "doc3"],
        "relevant_ids": ["doc1"],
        "relevance": {"doc1": 3},
    },
    {
        "input": "연차 몇 일까지 쓸 수 있나요?",
        "output": "연차는 총 15일이 제공되며 다음 해로 이월할 수 있습니다.",
        "expected": "15일, 이월 가능",
        "retrieved_context": ["연차는 연간 15일 제공됩니다.", "사용하지 않은 연차는 다음 해로 이월됩니다."],
        "retrieved_ids": ["doc5", "doc2", "doc9"],
        "relevant_ids": ["doc2", "doc5"],
        "relevance": {"doc5": 3, "doc2": 2},
    },
    {
        "input": "재택근무 신청은 어떻게 하나요?",
        "output": "재택근무는 팀장 승인 후 인사 시스템에서 신청합니다.",
        "expected": "팀장 승인 후 인사 시스템 신청",
        "retrieved_context": ["재택근무는 팀장 승인이 필요합니다.", "신청은 인사 시스템에서 진행합니다."],
        "retrieved_ids": ["doc8", "doc4"],
        "relevant_ids": ["doc1", "doc4"],
        "relevance": {"doc4": 2, "doc1": 3},
    },
]
for i, row in enumerate(DATASET):
    print(i, row["input"], "| retrieved:", row["retrieved_ids"], "| relevant:", row["relevant_ids"])

## 실제 judge LLM 설정

judge 기반 메트릭은 **실제 LLM**으로 채점한다. 공급자는 코드 인자가 아니라 환경변수
`AGENT_EVAL_JUDGE_PROVIDER` 하나로 전환하며(`anthropic` ↔ `openai`), 키는 레포 루트의
`.env` 파일에 넣는다(`.gitignore`에 의해 커밋되지 않음):

```
# .env 예시 — 둘 중 하나(또는 둘 다) 설정
AGENT_EVAL_JUDGE_PROVIDER=anthropic
ANTHROPIC_API_KEY=sk-ant-...

# AGENT_EVAL_JUDGE_PROVIDER=openai
# OPENAI_API_KEY=sk-...
```

이 규약(공급자 프리셋 포함)은 노트북 전용이 아니라 **서버 패키지(`agent_eval.server.judge`)의
공식 규약**이다 — 아래 셀은 서버가 시작할 때 부르는 바로 그 리졸버(`resolve_judge_from_env`)를
그대로 사용하므로, SDK 파트와 API 서버는 항상 같은 judge를 쓴다. 노트북 밖에서
`uv run agent-eval-serve`로 서버를 단독 실행해도 같은 `./.env`를 자동으로 읽는다.

공급자/모델을 바꾼 뒤에는 커널을 재시작한다(API 서버는 시작 시점의 설정을 계속 쓴다).
사내 게이트웨이·vLLM 등 임의의 OpenAI 호환 서버는 `AGENT_EVAL_JUDGE_BASE_URL`/`_MODEL`/
`_API_KEY`를 직접 지정하면 되고(프리셋보다 우선), Anthropic SDK를 직접 쓰는 팩토리 방식은
`AGENT_EVAL_JUDGE_FACTORY=examples/judges.py:claude_judge` 처럼 지정한다(최우선).

In [ ]:
# judge 설정은 서버와 완전히 같은 규약을 쓴다(agent_eval.server.judge — 서버 시작 시 부르는
# 바로 그 리졸버). 환경변수 하나로 실제 judge LLM 공급자를 전환한다:
#   AGENT_EVAL_JUDGE_PROVIDER=anthropic  →  Claude (ANTHROPIC_API_KEY 필요)
#   AGENT_EVAL_JUDGE_PROVIDER=openai     →  GPT    (OPENAI_API_KEY 필요)
import os
from pathlib import Path

from agent_eval.server.judge import load_env_file, resolve_judge_from_env

# 노트북 폴더에서 실행하든 레포 루트에서 실행하든 .env를 찾도록 둘 다 시도한다
# (이미 설정된 환경변수가 항상 우선한다).
load_env_file(Path.cwd() / ".env")
load_env_file(Path.cwd().parent / ".env")

# 실제 LLM 없이 스텁으로 조용히 떨어지지 않도록 미리 막고, 한국어로 안내한다.
if not (
    os.environ.get("AGENT_EVAL_JUDGE_FACTORY")
    or os.environ.get("AGENT_EVAL_JUDGE_BASE_URL")
    or os.environ.get("AGENT_EVAL_JUDGE_PROVIDER")
):
    raise RuntimeError(
        "실제 LLM judge 설정이 없다. 레포 루트의 .env(또는 셸)에 다음 중 하나를 설정한다:\n"
        "  AGENT_EVAL_JUDGE_PROVIDER=anthropic  (그리고 ANTHROPIC_API_KEY=...)\n"
        "  AGENT_EVAL_JUDGE_PROVIDER=openai     (그리고 OPENAI_API_KEY=...)\n"
        "  또는 AGENT_EVAL_JUDGE_BASE_URL / _MODEL / _API_KEY 직접 지정"
    )

resolved = resolve_judge_from_env()  # API 서버가 시작할 때 부르는 바로 그 리졸버
judge = resolved.backend             # SDK 파트에서 그대로 쓸 실제 LLM judge
print(f"judge: kind={resolved.kind}  detail={resolved.detail}")

---
# Part 1. SDK

judge는 위 설정 셀에서 만든 `judge`(서버와 같은 리졸버의 산출물)를 그대로 쓴다.

In [ ]:
from agent_eval.core.contracts import EvalContext, MetaKey
from agent_eval.metrics.rag import Faithfulness, NdcgAtK, PrecisionAtK, RecallAtK, ResponseConsistency

# 데이터셋 전체를 러너로 집계해 대표값과 95% 신뢰구간(CI)을 구하는 헬퍼.
# 각 항목은 '한 번만' 채점하고 그 결과를 러너에 재생(replay)해 집계한다 —
# judge 메트릭이 같은 항목으로 LLM을 두 번 호출하지 않도록, API 서버와 동일한 방식이다.
from types import SimpleNamespace

from agent_eval.core.gate import GatePolicy
from agent_eval.core.suite import Suite
from agent_eval.offline.runner import evaluate


def aggregate(metric, results, ctxs):
    replay = iter(results)
    shim = SimpleNamespace(  # 집계 속성만 흉내 내고, score()는 이미 계산한 결과를 돌려준다
        name=metric.name,
        requires=frozenset(),
        cost_class=metric.cost_class,
        aggregation=metric.aggregation,
        higher_is_better=metric.higher_is_better,
        unit_interval=getattr(metric, "unit_interval", True),
        score=lambda ctx: next(replay),
    )
    return evaluate(Suite("demo", metric.name, [shim], GatePolicy()), ctxs).aggregates[0]


def run_sdk(metric, ctxs):
    """각 항목을 개별 채점해 출력하고, 같은 결과로 데이터셋 집계를 출력한다."""
    print(f"[SDK] {metric.name}")
    results = [metric.score(ctx) for ctx in ctxs]
    for i, r in enumerate(results):
        print(f"  #{i}: score={r.score:.3f}  passed={r.passed}  error={r.error}")
        reason = (r.detail or {}).get("reason", "")
        if reason:
            print(f"      └ 판정 이유: {str(reason)[:110]}")
    agg = aggregate(metric, results, ctxs)
    print(f"  ▶ 집계 value={agg.value:.3f}  95% CI=[{agg.ci_low:.3f}, {agg.ci_high:.3f}]  n={agg.n}")

contexts = [
    EvalContext(
        input=row["input"],
        output=row["output"],
        expected=row["expected"],
        retrieved_context=row["retrieved_context"],
        metadata={
            MetaKey.RETRIEVED_IDS: row["retrieved_ids"],
            MetaKey.RELEVANT_IDS: row["relevant_ids"],
            MetaKey.RELEVANCE: row["relevance"],
        },
    )
    for row in DATASET
]
K = 3
print("준비 완료:", len(contexts), "개 컨텍스트, k =", K, ", judge =", resolved.detail)

## 1-1. `recall_at_k` — 재현율@k

정답(relevant) id 중 상위 k개 검색 결과에 들어온 비율. 검색기가 놓친 정답이 얼마나 되는지.

In [ ]:
run_sdk(RecallAtK(k=K), contexts)

## 1-2. `precision_at_k` — 정밀도@k

상위 k개 검색 결과 중 실제 정답인 비율. 검색 결과의 잡음 정도.

In [ ]:
run_sdk(PrecisionAtK(k=K), contexts)

## 1-3. `ndcg_at_k` — 정규화 DCG@k

정답을 더 위로 올릴수록 높은 점수. `metadata['relevance']`의 등급 관련도를 사용한다.

In [ ]:
run_sdk(NdcgAtK(k=K), contexts)

## 1-4. `faithfulness` — 근거 충실도

응답(`output`)의 모든 주장이 검색된 근거(`retrieved_context`)에서 뒷받침되는지 **실제 LLM judge**가 판단한다(할루시네이션 방지). `판정 이유`에서 근거를 확인한다.

In [ ]:
run_sdk(Faithfulness(judge), contexts)

## 1-5. `consistency` — 근거 일관성

응답이 근거와 **모순되지** 않는지 판단한다(누락은 무방, 모순만 감점). 충실도보다 관대한 기준이다.

In [ ]:
run_sdk(ResponseConsistency(judge), contexts)

---
# Part 2. API

health의 `judge` 항목이 실제 모델을 가리키는지 확인한다.

In [ ]:
# API 파트: 백그라운드 스레드에서 실제 FastAPI 서버를 띄우고, httpx로 진짜 HTTP 요청을 보낸다.
# 서버는 위 설정 셀이 만든 AGENT_EVAL_JUDGE_* 환경변수를 읽어 SDK 파트와 '같은' 실제 judge를
# 쓴다 — 아래 health 출력에서 judge.kind = openai_compatible, detail = 모델명으로 확인할 수 있다.
import threading
import time

import httpx
import uvicorn

from agent_eval.server.app import create_app

PORT = 8078
BASE_URL = f"http://127.0.0.1:{PORT}"

if "server" not in globals():
    server = uvicorn.Server(uvicorn.Config(create_app(), host="127.0.0.1", port=PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    while not server.started:
        time.sleep(0.1)
print("API 서버 준비 완료:", BASE_URL)
print("health:", httpx.get(f"{BASE_URL}/health").json())

def call_api(path, contexts, params=None):
    """엔드포인트에 contexts를 POST하고, 개별 결과와 집계를 출력한다."""
    payload = {"contexts": contexts}
    if params:
        payload["params"] = params
    resp = httpx.post(BASE_URL + path, json=payload, timeout=120)
    print(f"[API] POST {path} → {resp.status_code}")
    data = resp.json()
    if resp.status_code != 200:
        print("  오류:", data.get("detail"))
        return data
    for i, item in enumerate(data["results"]):
        print(f"  #{i}: score={item['score']:.3f}  passed={item['passed']}  error={item['error']}")
        reason = (item.get("detail") or {}).get("reason", "")
        if reason:
            print(f"      └ 판정 이유: {str(reason)[:110]}")
    agg = data["aggregate"]
    print(f"  ▶ 집계 value={agg['value']:.3f}  95% CI=[{agg['ci_low']:.3f}, {agg['ci_high']:.3f}]  n={agg['n']}")
    return data

## 2-1. `POST /rag/recall_at_k`

검색 메트릭은 `metadata`의 `retrieved_ids`·`relevant_ids`를 읽고, `params`로 k를 넘긴다. 순수 계산이므로 SDK와 **정확히** 일치한다.

In [ ]:
ctx = [{"metadata": {"retrieved_ids": r["retrieved_ids"], "relevant_ids": r["relevant_ids"]}} for r in DATASET]
call_api("/rag/recall_at_k", ctx, params={"k": K})

## 2-2. `POST /rag/precision_at_k`

In [ ]:
ctx = [{"metadata": {"retrieved_ids": r["retrieved_ids"], "relevant_ids": r["relevant_ids"]}} for r in DATASET]
call_api("/rag/precision_at_k", ctx, params={"k": K})

## 2-3. `POST /rag/ndcg_at_k`

등급 관련도를 쓰려면 `metadata['relevance']`도 함께 보낸다.

In [ ]:
ctx = [{"metadata": {"retrieved_ids": r["retrieved_ids"], "relevant_ids": r["relevant_ids"], "relevance": r["relevance"]}} for r in DATASET]
call_api("/rag/ndcg_at_k", ctx, params={"k": K})

## 2-4. `POST /rag/faithfulness`

응답(`output`)과 검색된 근거(`retrieved_context`)를 담는다. 같은 judge 설정이므로 SDK 1-4와 사실상 같은 점수가 나온다.

In [ ]:
ctx = [{"output": r["output"], "retrieved_context": r["retrieved_context"]} for r in DATASET]
call_api("/rag/faithfulness", ctx)

## 2-5. `POST /rag/consistency`

In [ ]:
ctx = [{"output": r["output"], "retrieved_context": r["retrieved_context"]} for r in DATASET]
call_api("/rag/consistency", ctx)